# 🧹 Notebook 2 — Data Cleaning & Feature Engineering
**Project:** Customer Churn Prediction  
**Author:** Aditya Bobade

---
### Objectives
- Handle missing values and data types
- Encode categorical variables
- Engineer new features (TenureGroup, NumServices, ChargePerTenure, HighValue)
- Handle class imbalance via oversampling
- Save cleaned datasets for modelling


In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")

df = pd.read_csv('../data/telco_churn.csv')
print("Original shape:", df.shape)


Original shape: (7043, 21)


In [2]:
# Step 1: Drop non-predictive ID column
df.drop(columns=['customerID'], inplace=True)

# Step 2: Fix TotalCharges type
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)

# Step 3: Encode target
df['Churn'] = (df['Churn'] == 'Yes').astype(int)
print("Churn distribution:\n", df['Churn'].value_counts())


Churn distribution:
 Churn
0    5011
1    2032
Name: count, dtype: int64


In [3]:
# Step 4: Feature Engineering
df['TenureGroup']    = pd.cut(df['tenure'], bins=[0,12,24,48,72],
                               labels=['0-1yr','1-2yr','2-4yr','4+yr'])
df['ChargePerTenure'] = (df['MonthlyCharges'] / (df['tenure'] + 1)).round(2)

service_cols = ['OnlineSecurity','OnlineBackup','DeviceProtection',
                'TechSupport','StreamingTV','StreamingMovies']
df['NumServices'] = df[service_cols].apply(lambda r: sum(v=='Yes' for v in r), axis=1)
df['HighValue']   = (df['MonthlyCharges'] > df['MonthlyCharges'].quantile(0.75)).astype(int)

print("New features added: TenureGroup, ChargePerTenure, NumServices, HighValue")
print(df[['tenure','TenureGroup','NumServices','HighValue','ChargePerTenure']].head())


New features added: TenureGroup, ChargePerTenure, NumServices, HighValue
   tenure TenureGroup  NumServices  HighValue  ChargePerTenure
0      63        4+yr            1          0             1.31
1       3       0-1yr            0          0             6.34
2      42       2-4yr            1          0             1.31
3      22       1-2yr            0          0             0.90
4      37       2-4yr            0          0             0.48


In [4]:
# Step 5: Encode categoricals
from sklearn.preprocessing import LabelEncoder

for col in ['gender','Partner','Dependents','PhoneService','PaperlessBilling']:
    df[col] = LabelEncoder().fit_transform(df[col])

multi_cols = ['MultipleLines','InternetService','OnlineSecurity','OnlineBackup',
              'DeviceProtection','TechSupport','StreamingTV','StreamingMovies',
              'Contract','PaymentMethod','TenureGroup']
df = pd.get_dummies(df, columns=multi_cols, drop_first=True)
print("Shape after encoding:", df.shape)


Shape after encoding: (7043, 37)


In [5]:
# Step 6: Class imbalance — oversample minority class
from sklearn.utils import resample

majority   = df[df['Churn']==0]
minority   = df[df['Churn']==1]
print(f"Before — Majority: {len(majority)}, Minority: {len(minority)}")

minority_up = resample(minority, replace=True, n_samples=len(majority), random_state=42)
df_balanced = pd.concat([majority, minority_up]).sample(frac=1, random_state=42)
print(f"After  — Total: {len(df_balanced)}, Balance: {df_balanced['Churn'].value_counts().to_dict()}")


Before — Majority: 5011, Minority: 2032
After  — Total: 10022, Balance: {0: 5011, 1: 5011}


In [6]:
# Step 7: Correlation heatmap
top_feats = ['tenure','MonthlyCharges','TotalCharges','NumServices','ChargePerTenure','HighValue','Churn']
corr = df[top_feats].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            mask=mask, ax=ax, linewidths=0.5)
ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../charts/06_correlation_heatmap.png', bbox_inches='tight')
plt.show()


In [7]:
# Save cleaned datasets
df.to_csv('../data/telco_churn_cleaned.csv', index=False)
df_balanced.to_csv('../data/telco_churn_balanced.csv', index=False)
print("Saved: telco_churn_cleaned.csv  —", df.shape)
print("Saved: telco_churn_balanced.csv —", df_balanced.shape)


Saved: telco_churn_cleaned.csv  — (7043, 37)
Saved: telco_churn_balanced.csv — (10022, 37)
